In [0]:
# Instance audit: projects → Snowflake/OpenAI/code envs

This notebook scans the current DSS instance and writes a dataset named `INSTANCE_PROJECT_USAGE` in this project.

- Read-only against scanned projects
- Requires permissions to list/access the projects

In [0]:
import pandas as pd
import dataiku

from dku_project_bulk_move.instance_audit import collect_instance_project_usage

OUTPUT_DATASET = "INSTANCE_PROJECT_USAGE"

client = dataiku.api_client()
project_key = dataiku.default_project_key()
project = client.get_project(project_key)

# Ensure the output dataset exists
try:
    project.get_dataset(OUTPUT_DATASET).get_settings()
except Exception:
    try:
        project.create_dataset(
            OUTPUT_DATASET,
            type="Filesystem",
            params={"connection": "filesystem_managed", "path": OUTPUT_DATASET},
            formatType="csv",
            formatParams={"separator": ",", "style": "excel"},
        )
    except Exception:
        project.create_dataset(OUTPUT_DATASET, type="Inline", params={})

# Collect + write
rows = collect_instance_project_usage(client)
df = pd.DataFrame.from_records([r.as_dict() for r in rows])
dataiku.Dataset(OUTPUT_DATASET).write_with_schema(df)

{"rows": len(df), "dataset": OUTPUT_DATASET, "projectKey": project_key}